Bước 1: Load dữ liệu

In [ ]:
import joblib
import numpy as np
import tensorflow as tf
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Dense, Dropout, BatchNormalization
from tensorflow.keras.optimizers import Adam
import os
from google.colab import drive

# =============================================================================
# 1. KẾT NỐI DRIVE & CẤU HÌNH ĐƯỜNG DẪN
# =============================================================================
try:
    drive.mount('/content/drive')
except:
    pass

# Đường dẫn đến thư mục chứa dữ liệu Đa lớp (Anh kiểm tra lại nếu khác)
BASE_DIR = "/content/drive/MyDrive/DoAn_NIDS/Dataset/"
DATA_PATH = os.path.join(BASE_DIR, "Multi_Data/")

print("-" * 60)
print(f"📂 Đang đọc dữ liệu từ: {DATA_PATH}")

# =============================================================================
# 2. LOAD DỮ LIỆU ĐA LỚP
# =============================================================================
# Lưu ý: Các file này là kết quả của bước Tiền xử lý (đã One-Hot và Cân bằng)
print("⏳ Đang load các file .pkl (X_train, y_train, X_test, y_test)...")

try:
    X_train = joblib.load(DATA_PATH + 'X_train_multi.pkl')
    y_train = joblib.load(DATA_PATH + 'y_train_multi.pkl') # Nhãn dạng One-Hot
    X_test = joblib.load(DATA_PATH + 'X_test_multi.pkl')
    y_test = joblib.load(DATA_PATH + 'y_test_multi.pkl')   # Nhãn dạng One-Hot

    # Load trọng số lớp (Class Weights) để Model chú ý nhóm hiếm
    class_weights = joblib.load(DATA_PATH + 'class_weights_multi.pkl')

    print("✅ Load dữ liệu thành công!")
except FileNotFoundError as e:
    print(f"❌ LỖI: Không tìm thấy file. Vui lòng kiểm tra lại đường dẫn!\n{e}")

# =============================================================================
# 3. KIỂM TRA KÍCH THƯỚC DỮ LIỆU
# =============================================================================
# Lấy thông số tự động từ dữ liệu
input_dim = X_train.shape[1]   # Số đặc trưng đầu vào (VD: 37)
n_classes = y_train.shape[1]   # Số lớp đầu ra (Bắt buộc phải là 5)

print("-" * 60)
print(f"📊 THÔNG SỐ DỮ LIỆU:")
print(f"   - Input Dimension (Số cột feature): {input_dim}")
print(f"   - Number of Classes (Số lớp):       {n_classes}") # Kỳ vọng: 5
print(f"   - Số lượng mẫu Train:               {len(X_train)}")
print(f"   - Trọng số lớp (Class Weights):     {class_weights}")
print("-" * 60)


Bước 2: Dựng khung mô hình

In [ ]:
# =============================================================================
# 4. XÂY DỰNG MÔ HÌNH DNN (MULTICLASS)
# =============================================================================
def build_multiclass_dnn():
    model = Sequential(name="DNN_Multiclass_5_Classes")

    # --- Hidden Layer 1 ---
    model.add(Dense(512, input_shape=(input_dim,), activation='relu'))
    model.add(BatchNormalization())
    model.add(Dropout(0.2))

    # --- Hidden Layer 2 ---
    model.add(Dense(256, activation='relu'))
    model.add(BatchNormalization())
    model.add(Dropout(0.25))

    # --- Hidden Layer 3 ---
    model.add(Dense(128, activation='relu'))
    model.add(BatchNormalization())
    model.add(Dropout(0.25))

    # --- Output Layer (QUAN TRỌNG) ---
    # Phân loại 5 lớp -> Dùng 5 nơ-ron và hàm Softmax
    model.add(Dense(n_classes, activation='softmax'))

    # Compile mô hình
    # - Loss: dùng 'categorical_crossentropy' vì nhãn là One-Hot
    # - Metric: vẫn dùng 'accuracy'
    optimizer = Adam(learning_rate=0.001)
    model.compile(loss='categorical_crossentropy',
                  optimizer=optimizer,
                  metrics=['accuracy'])

    return model

# Khởi tạo và xem kiến trúc
if 'n_classes' in locals() and n_classes == 5:
    model_multi = build_multiclass_dnn()
    model_multi.summary()
else:
    print("⚠️ CẢNH BÁO: Số lượng lớp không đúng 5. Vui lòng kiểm tra lại bước Tiền xử lý.")

In [ ]:
from tensorflow.keras.callbacks import ModelCheckpoint, EarlyStopping, ReduceLROnPlateau
import matplotlib.pyplot as plt
import os

# =============================================================================
# 1. CẤU HÌNH CALLBACKS (CÁC GIÁM SÁT VIÊN)
# =============================================================================
print("-" * 60)
print("👮 ĐANG THIẾT LẬP CÁC GIÁM SÁT VIÊN (CALLBACKS)...")

# Đường dẫn để lưu model tốt nhất
# File .keras là định dạng mới chuẩn của Keras (nhẹ và nhanh hơn .h5)
checkpoint_path = os.path.join(BASE_DIR, 'Scenario2_Multiclass_Best.keras')

callbacks = [
    # 1. Lưu lại model có val_loss thấp nhất (tránh lưu model bị overfitting)
    ModelCheckpoint(
        filepath=checkpoint_path,
        monitor='val_loss',
        save_best_only=True,
        mode='min',
        verbose=1
    ),

    # 2. Dừng sớm nếu không cải thiện sau 10 vòng (tránh lãng phí thời gian)
    EarlyStopping(
        monitor='val_loss',
        patience=10,
        restore_best_weights=True,
        verbose=1
    ),

    # 3. Giảm tốc độ học nếu loss đi ngang (giúp model len lỏi vào điểm tối ưu)
    ReduceLROnPlateau(
        monitor='val_loss',
        factor=0.5,      # Giảm một nửa tốc độ
        patience=3,      # Chờ 3 vòng không giảm
        min_lr=0.00001,  # Không giảm thấp hơn mức này
        verbose=1
    )
]

print(f"✅ Model sẽ được lưu tại: {checkpoint_path}")

# =============================================================================
# 2. THỰC HIỆN HUẤN LUYỆN (TRAINING)
# =============================================================================
print("-" * 60)
print("🚀 BẮT ĐẦU HUẤN LUYỆN (Quá trình này có thể tốn vài phút)...")
print("   - Batch Size: 256")
print("   - Epochs: 50 (Sẽ dừng sớm nếu cần)")
print("   - Class Weights: Đã kích hoạt")

# Lưu ý: model_multi là biến model mình đã tạo ở Bước 1
if 'model_multi' in locals():
    history = model_multi.fit(
        X_train, y_train,
        validation_split=0.1,       # Dùng 10% dữ liệu train để kiểm tra chéo ngay lúc học
        epochs=50,                  # Số vòng lặp tối đa
        batch_size=256,             # Số lượng mẫu học mỗi lần cập nhật trọng số
        class_weight=class_weights, # QUAN TRỌNG: Giúp cân bằng sự chú ý cho các nhóm hiếm
        callbacks=callbacks,
        verbose=1
    )
    print("\n✅ HUẤN LUYỆN HOÀN TẤT!")
else:
    print("❌ LỖI: Không tìm thấy biến 'model_multi'. Hãy chạy lại Bước 1 trước nhé!")

# =============================================================================
# 3. VẼ BIỂU ĐỒ ĐÁNH GIÁ (LOSS & ACCURACY)
# =============================================================================
if 'history' in locals():
    plt.figure(figsize=(14, 6))

    # --- Biểu đồ Loss (Hàm mất mát) ---
    plt.subplot(1, 2, 1)
    plt.plot(history.history['loss'], label='Train Loss', color='blue', linewidth=2)
    plt.plot(history.history['val_loss'], label='Val Loss', color='orange', linewidth=2)
    plt.title('Hàm mất mát (Loss) - Càng thấp càng tốt')
    plt.xlabel('Vòng (Epochs)')
    plt.ylabel('Loss')
    plt.legend()
    plt.grid(True)

    # --- Biểu đồ Accuracy (Độ chính xác) ---
    plt.subplot(1, 2, 2)
    plt.plot(history.history['accuracy'], label='Train Acc', color='green', linewidth=2)
    plt.plot(history.history['val_accuracy'], label='Val Acc', color='red', linewidth=2)
    plt.title('Độ chính xác (Accuracy) - Càng cao càng tốt')
    plt.xlabel('Vòng (Epochs)')
    plt.ylabel('Accuracy')
    plt.legend()
    plt.grid(True)

    plt.show()

In [ ]:
from sklearn.metrics import classification_report, confusion_matrix
import matplotlib.pyplot as plt
import seaborn as sns
import numpy as np
import time

# =============================================================================
# BƯỚC 3: ĐÁNH GIÁ MÔ HÌNH TRÊN TẬP TEST
# =============================================================================
if 'model_multi' in locals():
    print("-" * 60)
    print("🧐 ĐANG CHẤM ĐIỂM MÔ HÌNH TRÊN TẬP TEST...")

    # 1. Dự đoán
    # Kết quả trả về là xác suất (VD: [0.1, 0.8, 0.05, 0.05, 0.0])
    start_time = time.time()
    y_pred_probs = model_multi.predict(X_test)
    end_time = time.time()

    # 2. Chuyển về nhãn (Lấy vị trí có xác suất cao nhất)
    # VD: [0.1, 0.8, ...] -> Nhãn 1
    y_pred = np.argmax(y_pred_probs, axis=1)

    # Vì y_test ban đầu mình load là dạng One-Hot, cần chuyển về dạng số nguyên để so sánh
    y_true = np.argmax(y_test, axis=1)

    # 3. Tên các lớp (Theo đúng thứ tự mapping lúc đầu)
    class_names = ['Benign', 'DoS/DDoS', 'PortScan', 'BruteForce', 'Other']

    # --- A. Báo cáo chi tiết ---
    print("\n📊 CLASSIFICATION REPORT:")
    print(classification_report(y_true, y_pred, target_names=class_names, digits=4))

    # --- B. Ma trận nhầm lẫn (Confusion Matrix) ---
    cm = confusion_matrix(y_true, y_pred)

    plt.figure(figsize=(10, 8))
    sns.heatmap(cm, annot=True, fmt='d', cmap='Blues',
                xticklabels=class_names,
                yticklabels=class_names)
    plt.xlabel('Dự đoán (Predicted)')
    plt.ylabel('Thực tế (Actual)')
    plt.title('Confusion Matrix - Multiclass DNN')
    plt.show()

    # --- C. Tốc độ ---
    print(f"⏱️ Thời gian xử lý trung bình: {(end_time - start_time)/len(y_test) * 1000000:.2f} µs/mẫu")

else:
    print("❌ Lỗi: Không tìm thấy biến 'model_multi'.")